# Detección de congestión en un enlace con árboles de decisión

**Escenario.** Dos equipos, A (emisor, `192.168.50.10`) y B (receptor, `192.168.50.20`), se comunican por un enlace cuya salida en A está limitada a **10 Mbit/s** con una cola TBF de 32 kB. Con `iperf3` se generó tráfico UDP en fases aleatorias (2–16 Mbit/s, 5–15 s, datagramas de 400–1400 B) y se capturó la traza en el receptor B.

**Registro.** Cada fila de `traza.csv` es **un paquete** recibido en B.

**Definición de congestión.** Un paquete está *congestionado* si su **retardo de A a B supera 5 ms**. Sin carga, el retardo es de unos 0,01 ms; con la cola llena, de unos 26 ms (32 kB a 10 Mbit/s). Con 5 ms de retardo hay unos 6 kB en cola: la cola ya se está acumulando, aunque todavía no se desborde.

**Objetivo.** Entrenar un **árbol de decisión** que clasifique si el enlace está congestionado **en este momento**, usando solo lo que se observa en el receptor (tiempos de llegada, tasas y pérdidas). El retardo **no** se usa como entrada, porque define la etiqueta.

## 1. Librerías y carga de datos

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)

In [ ]:
# En Colab: si el archivo no está en la sesión, se abre el diálogo para subirlo
# (se recomienda subir traza.csv.gz, que pesa ~6 veces menos que traza.csv).
candidatos = ["traza.csv.gz", "traza.csv"]
if not any(os.path.exists(c) for c in candidatos):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError("Coloca traza.csv.gz o traza.csv junto a este notebook.")

ARCHIVO = next(c for c in candidatos if os.path.exists(c))
df = pd.read_csv(ARCHIVO).sort_values("tiempo_s", ignore_index=True)
print(f"Archivo: {ARCHIVO}")
print(f"Registros (paquetes): {len(df):,}   |   duración: {df['tiempo_s'].iat[-1]/60:.1f} min")
df.head()

## 2. Exploración

In [ ]:
print("Balance de clases:")
balance = df["congestion"].value_counts().rename({0: "normal (0)", 1: "congestionado (1)"})
print(pd.DataFrame({"paquetes": balance, "%": (balance / len(df) * 100).round(1)}))
df.describe().T.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(df["retardo_ms"].clip(lower=1e-3), bins=np.logspace(-3, 1.6, 120), color="#2c7fb8")
ax.axvline(5, color="#d7301f", ls="--", label="umbral de congestión = 5 ms")
ax.set_xscale("log")
ax.set_xlabel("retardo A → B (ms, escala log)")
ax.set_ylabel("paquetes")
ax.set_title("Distribución del retardo: dos modos bien separados (cola vacía vs. cola llena)")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Dos minutos de la traza para ver la relación entre carga, retardo y etiqueta
s = df[(df["tiempo_s"] >= 60) & (df["tiempo_s"] < 180)]
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].plot(s["tiempo_s"], s["tasa_100ms_mbit"], lw=0.7, color="#2c7fb8", label="tasa recibida (100 ms)")
ax[0].step(s["tiempo_s"], s["info_tasa_ofrecida_mbit"], lw=1, ls="--", color="0.5", label="tasa ofrecida por iperf3 (solo referencia)")
ax[0].axhline(10, color="#d7301f", lw=0.8, ls=":", label="límite 10 Mbit/s")
ax[0].set_ylabel("Mbit/s"); ax[0].legend(fontsize=8, loc="upper left", ncol=3)
ax[1].plot(s["tiempo_s"], s["retardo_ms"], lw=0.7, color="#636363")
ax[1].axhline(5, color="#d7301f", ls="--", lw=0.8)
ax[1].set_ylabel("retardo (ms)")
ax[2].fill_between(s["tiempo_s"], s["congestion"], step="post", color="#e6550d", alpha=0.6)
ax[2].set_ylabel("congestión"); ax[2].set_yticks([0, 1]); ax[2].set_xlabel("tiempo (s)")
fig.suptitle("Tasa recibida, retardo y etiqueta de congestión")
plt.tight_layout(); plt.show()

## 3. Atributos y etiqueta

| Atributo | Qué mide |
|---|---|
| `tamano_b` | longitud de trama del paquete |
| `iat_ms` | tiempo entre este paquete y el anterior (entre-llegadas) |
| `iat_media_50`, `iat_std_50` | media y desviación (jitter) de las entre-llegadas en los últimos 50 paquetes |
| `tasa_100ms_mbit`, `paquetes_100ms` | Mbit/s y paquetes recibidos en los últimos 100 ms |
| `tasa_1s_mbit` | Mbit/s recibidos en el último segundo |
| `salto_seq`, `perdidas_100ms` | paquetes perdidos (saltos en la secuencia de iperf3) justo antes y en los últimos 100 ms |

Se **excluyen**: `retardo_ms`, porque define la etiqueta; `info_*`, porque es el calendario de iperf3 y en una red real no se conoce; y `tiempo_s`, `flujo` y `seq`, porque son identificadores que no generalizan.

In [ ]:
ATRIBUTOS = ["tamano_b", "iat_ms", "iat_media_50", "iat_std_50",
             "tasa_100ms_mbit", "paquetes_100ms", "tasa_1s_mbit",
             "salto_seq", "perdidas_100ms"]
X = df[ATRIBUTOS]
y = df["congestion"]
X.corrwith(y).sort_values(key=abs, ascending=False).round(3).to_frame("correlación con congestion")

## 4. Partición temporal: entrenamiento 60 % / validación 20 % / prueba 20 %

Paquetes vecinos son casi idénticos. Si se mezclaran al azar, el modelo vería en entrenamiento "copias" de los paquetes de prueba y las métricas saldrían infladas. Por eso la traza se corta **en orden de tiempo**:

- **Entrenamiento** (primer 60 %): se ajustan los árboles.
- **Validación** (siguiente 20 %): se elige el mejor árbol.
- **Prueba** (último 20 %): se evalúa **una sola vez** el árbol elegido.

In [ ]:
c1, c2 = int(len(df) * 0.60), int(len(df) * 0.80)
X_train, X_val, X_test = X.iloc[:c1], X.iloc[c1:c2], X.iloc[c2:]
y_train, y_val, y_test = y.iloc[:c1], y.iloc[c1:c2], y.iloc[c2:]
for nombre, a, b, yy in (("Entrenamiento", 0, c1, y_train), ("Validación", c1, c2, y_val), ("Prueba", c2, len(df), y_test)):
    print(f"{nombre:14s} {len(yy):>8,} paquetes  ({df['tiempo_s'].iat[a]:5.0f} – {df['tiempo_s'].iat[b-1]:5.0f} s)"
          f"  | congestionados {yy.mean()*100:.1f} %")

## 5. Selección del árbol con el conjunto de validación

Se entrenan tres candidatos con **profundidad máxima 1, 3 y sin límite**, todos con Gini, `class_weight="balanced"` (para compensar el desbalance de clases) y `random_state=42`. Cada uno se ajusta **solo con entrenamiento** y se compara en **validación**.

Criterio de selección: el mayor **F1 de la clase *congestionado*** en validación. Si hay empate, se prefiere el de menor profundidad efectiva, luego el de menos hojas y, por último, el primero en el orden 1, 3, sin límite.

In [ ]:
def metricas(y_true, y_hat):
    return {"accuracy": accuracy_score(y_true, y_hat),
            "precisión": precision_score(y_true, y_hat, zero_division=0),
            "sensibilidad": recall_score(y_true, y_hat, zero_division=0),
            "F1": f1_score(y_true, y_hat, zero_division=0)}

candidatos = {}
filas = []
for orden, prof in enumerate([1, 3, None]):
    arbol = DecisionTreeClassifier(criterion="gini", max_depth=prof,
                                   class_weight="balanced", random_state=42).fit(X_train, y_train)
    nombre = f"max_depth={prof}"
    candidatos[nombre] = arbol
    pred_val = arbol.predict(X_val)
    filas.append({"candidato": nombre,
                  "accuracy entrenamiento": accuracy_score(y_train, arbol.predict(X_train)),
                  "accuracy validación": accuracy_score(y_val, pred_val),
                  "F1 validación": f1_score(y_val, pred_val, zero_division=0),
                  "profundidad efectiva": arbol.get_depth(),
                  "hojas": arbol.get_n_leaves(),
                  "orden": orden})

tabla = pd.DataFrame(filas)
elegido = tabla.sort_values(["F1 validación", "profundidad efectiva", "hojas", "orden"],
                            ascending=[False, True, True, True]).iloc[0]["candidato"]
modelo = candidatos[elegido]
print(f"Modelo seleccionado con validación: {elegido}")
tabla.drop(columns="orden").round(4)

## 6. Evaluación en prueba (último 20 % de la traza, una sola vez)

In [ ]:
y_pred = modelo.predict(X_test)
print(f"Árbol elegido: {elegido} → profundidad {modelo.get_depth()}, {modelo.get_n_leaves()} hojas\n")
print(classification_report(y_test, y_pred, target_names=["normal", "congestionado"], digits=4, zero_division=0))
pd.Series(metricas(y_test, y_pred)).round(4).to_frame(f"prueba ({elegido})")

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["normal", "congestionado"]).plot(ax=ax, cmap="Blues", values_format=",")
ax.set_title("Matriz de confusión (prueba)")
plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(modelo.feature_importances_, index=ATRIBUTOS).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.6))
imp.plot.barh(ax=ax, color="#2c7fb8")
ax.set_xlabel("importancia (reducción de impureza Gini)")
ax.set_title("Importancia de los atributos en el árbol seleccionado")
plt.tight_layout(); plt.show()

## 7. Árbol pequeño para interpretar las reglas

El árbol seleccionado puede ser demasiado grande para leerlo. Uno de **profundidad 3** muestra las reglas principales y permite ver cuánto rendimiento se pierde a cambio de poder interpretarlo.

In [ ]:
arbol3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_train, y_train)
pred3 = arbol3.predict(X_test)
comparacion = pd.DataFrame({f"seleccionado ({elegido})": metricas(y_test, y_pred),
                            "árbol profundidad 3": metricas(y_test, pred3)})
print(export_text(arbol3, feature_names=ATRIBUTOS, decimals=3))
comparacion.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(arbol3, feature_names=ATRIBUTOS, class_names=["normal", "congestionado"],
          filled=True, rounded=True, impurity=False, proportion=True, precision=3, fontsize=9, ax=ax)
ax.set_title("Árbol de decisión de profundidad 3")
plt.tight_layout(); plt.show()

## 8. Predicción en el tiempo y análisis de errores

In [ ]:
prueba = df.iloc[c2:].copy()
prueba["pred"] = y_pred
prueba["error"] = np.where(prueba["pred"] == prueba["congestion"], "acierto",
                  np.where(prueba["pred"] == 1, "falsa alarma", "omisión"))

t0 = prueba["tiempo_s"].iat[0]
s = prueba[(prueba["tiempo_s"] >= t0) & (prueba["tiempo_s"] < t0 + 90)]
fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].plot(s["tiempo_s"], s["tasa_100ms_mbit"], lw=0.7, color="#2c7fb8")
ax[0].axhline(10, color="#d7301f", lw=0.8, ls=":")
ax[0].set_ylabel("tasa 100 ms\n(Mbit/s)")
ax[1].plot(s["tiempo_s"], s["retardo_ms"], lw=0.7, color="#636363")
ax[1].axhline(5, color="#d7301f", ls="--", lw=0.8)
ax[1].set_ylabel("retardo (ms)")
ax[2].fill_between(s["tiempo_s"], s["congestion"] * 1.0, step="post", color="#e6550d", alpha=0.45, label="real")
ax[2].step(s["tiempo_s"], s["pred"] * 0.8, where="post", color="#252525", lw=1, label="predicho (×0,8)")
e = s[s["error"] != "acierto"]
ax[2].scatter(e["tiempo_s"], np.full(len(e), 0.5), s=6, color="#d7301f", label="error")
ax[2].set_yticks([0, 1]); ax[2].set_ylabel("congestión"); ax[2].set_xlabel("tiempo (s)")
ax[2].legend(fontsize=8, loc="upper right", ncol=3)
fig.suptitle("Primeros 90 s de la prueba: real vs. predicho")
plt.tight_layout(); plt.show()

In [ ]:
# ¿Dónde se equivoca? Por tasa ofrecida (solo para interpretar) y por cercanía a un cambio de estado
print("Errores por tasa ofrecida por iperf3:")
print(prueba.groupby("info_tasa_ofrecida_mbit")["error"].value_counts().unstack(fill_value=0))

cambio = prueba["congestion"].ne(prueba["congestion"].shift()).cumsum()
t_cambio = prueba.groupby(cambio)["tiempo_s"].transform("min")
prueba["s_desde_cambio"] = prueba["tiempo_s"] - t_cambio
prueba["zona"] = np.where(prueba["s_desde_cambio"] < 0.2, "< 200 ms de una transición", "régimen estable")
print("\nTasa de error según cercanía a una transición real normal ↔ congestionado:")
print(prueba.groupby("zona")["error"].apply(lambda e: (e != "acierto").mean()).map("{:.2%}".format))

## 9. Conclusiones

**Rendimiento.** El árbol seleccionado con validación (`max_depth=None`, profundidad efectiva 15, 102 hojas) obtuvo en prueba, evaluado una sola vez sobre 144.738 paquetes que no se usaron en el ajuste, una **exactitud de 0,9936**, una **precisión de 0,9895**, una **sensibilidad de 0,9980** y un **F1 de 0,9937** para la clase *congestionado*. La matriz de confusión registra 73.980 VP, 69.827 VN, **782 falsas alarmas (FP)** y **149 omisiones (FN)**. En este problema, una omisión es más grave que una falsa alarma: no detectar una congestión deja pasar el aumento de retardo y las pérdidas. El modelo comete pocas omisiones, por eso su sensibilidad es más alta que su precisión.

**Selección y sobreajuste.** Los tres candidatos quedaron muy parejos en validación (F1 de 0,9985, 0,9973 y 0,9990 para profundidad 1, 3 y sin límite). Por la regla de selección ganó el árbol sin límite, aunque supera por solo 0,0005 al árbol de profundidad 1, que tiene **2 hojas en lugar de 102**. El árbol sin límite muestra señales de sobreajuste: su exactitud de entrenamiento es 1,0000, memoriza el entrenamiento, y en prueba el árbol de profundidad 3 lo superó (F1 0,9953 frente a 0,9937). Siguiendo el protocolo, la selección no se cambió después de ver la prueba. Con un empate tan estrecho, un criterio de simplicidad habría preferido un árbol más pequeño, más fácil de interpretar y con un resultado prácticamente igual.

**Atributos clave.** El árbol se apoya casi por completo en `tasa_100ms_mbit`, con una importancia de 0,989; el resto de atributos aporta menos de 0,004 cada uno. El árbol de profundidad 1 se resume en una sola regla: **SI la tasa recibida en los últimos 100 ms es mayor que 9,833 Mbit/s ENTONCES está congestionado**. Tiene sentido físico: cuando la cola se llena, el cuello de botella entrega paquetes exactamente al ritmo del límite de 10 Mbit/s, así que la tasa recibida se queda fija en ese valor. Sin congestión, en cambio, la tasa recibida es igual a la ofrecida, que siempre está por debajo del límite.

**Errores.** Los errores se concentran en las **transiciones entre estados**: la tasa de error es de 11,6 % en los 200 ms cercanos a un cambio entre normal y congestionado, frente a 0,43 % en régimen estable. Los atributos usan ventanas de 100 ms a 1 s y reaccionan con retraso, mientras que la cola se llena o se vacía en milisegundos. También aparecen más falsas alarmas con 10 Mbit/s ofrecidos (621), justo en el límite, donde la tasa recibida queda muy cerca del umbral de 9,833 Mbit/s.

**Limitaciones.**
- La etiqueta depende del umbral de 5 ms; otro umbral cambiaría la proporción de paquetes congestionados.
- El enlace es emulado: dos *network namespaces* conectados por veth en un solo equipo, con un único flujo UDP y una cola TBF fija. En una red real, con tráfico TCP, varios flujos y capacidad variable, la tasa recibida no se fijaría en un valor conocido y el problema sería más difícil.
- La proporción de congestión cambia entre particiones (64 % en entrenamiento, 73 % en validación, 51 % en prueba) porque las fases son aleatorias.
- El modelo clasifica el **estado actual**; no anticipa una congestión futura.
- La traza viene de una sola captura de 11 minutos. Para usar el modelo fuera del taller habría que validarlo con otras capturas, otros límites de enlace y tráfico real.